# Custom Oriented (Rotated) Detection Training

Train a custom oriented (rotated) bounding-box detector on your Roboflow dataset.

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

**After running the install cell, RESTART the runtime** (Runtime > Restart session), then run all cells from the top.

## 1. Download Dataset from Roboflow

Export your Roboflow project using the **Oriented Bounding Boxes** annotation type.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("coco")

## 2. Install Dependencies

**After this cell finishes, RESTART the runtime (Runtime > Restart session) then run all cells from the top again.**

In [ ]:
try:
    import setuptools
    sv = int(setuptools.__version__.split('.')[0])
    assert 70 <= sv < 80, f"setuptools must be 70<=v<80, got {setuptools.__version__}"
    import pkg_resources
    import numpy as np
    assert np.__version__.startswith("1."), f"numpy must be <2, got {np.__version__}"
    import torch, mmcv, mmengine, mmdet, mmrotate
    import openvino as ov
    import nncf
    from mmcv.ops import MultiScaleDeformableAttention
    assert torch.__version__.startswith("2.3")
    assert mmcv.__version__.startswith("2.2")
    print(f"Already installed:")
    print(f"  setuptools: {setuptools.__version__}")
    print(f"  numpy:    {np.__version__}")
    print(f"  torch:    {torch.__version__}")
    print(f"  mmcv:     {mmcv.__version__}")
    print(f"  mmengine: {mmengine.__version__}")
    print(f"  mmdet:    {mmdet.__version__}")
    print(f"  mmrotate: {mmrotate.__version__}")
    print(f"  openvino: {ov.__version__}")
    print(f"  nncf:     {nncf.__version__}")
except (ImportError, AssertionError, ModuleNotFoundError):
    print("Installing (first time, ~3-4 min)...")
    
    # Remove broken system pkg_resources (blocks pip's setuptools on Python 3.12)
    !rm -rf /usr/lib/python3/dist-packages/pkg_resources
    
    # Clean slate
    !pip uninstall -y torch torchvision torchaudio mmcv mmcv-lite mmengine mmdet mmrotate openxlab --quiet
    
    # Torch 2.3.0 + cu121
    !pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 --quiet
    
    # mmcv 2.2.0 (only combo with Python 3.12 prebuilt wheels)
    !pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html --quiet
    
    # mmdet 3.3.0 + mmrotate dev-1.x (compatible with mmcv 2.x)
    !pip install mmengine mmdet==3.3.0 --quiet
    !pip install "git+https://github.com/open-mmlab/mmrotate.git@dev-1.x" --quiet
    
    # Patch mmdet + mmrotate hardcoded mmcv version checks
    !sed -i "s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/" /usr/local/lib/python3.12/dist-packages/mmdet/__init__.py
    !sed -i "s/mmcv_maximum_version = '2.1.0'/mmcv_maximum_version = '2.3.0'/" /usr/local/lib/python3.12/dist-packages/mmrotate/__init__.py
    !sed -i "s/mmdet_maximum_version = '3.2.0'/mmdet_maximum_version = '3.4.0'/" /usr/local/lib/python3.12/dist-packages/mmrotate/__init__.py
    
    # OpenVINO + NNCF + ONNX export
    !pip install openvino nncf onnx onnxruntime --quiet
    
    # CRITICAL: force these LAST so nothing downgrades them
    !pip uninstall -y openxlab --quiet
    !pip install --force-reinstall "setuptools>=70,<80" --quiet
    !pip install "numpy<2" --quiet
    
    print("\n>>> RESTART RUNTIME and run all cells from the top <<<")

## 3. Training Parameters

Adjust `MODEL_SIZE`, `EPOCHS`, `OPTIMIZE`, and `AUG` as needed. The augmentation dict controls training-time data augmentation.

In [ ]:
import os
import json
import shutil
import pickle
import glob
import numpy as np
import torch
from pathlib import Path
from google.colab import files

# ============================================================
# EDIT THIS SECTION
# ============================================================
MODEL_SIZE = "tiny"        # "tiny" | "small" | "medium" | "large"
EPOCHS = 150
BATCH_SIZE = 16
IMAGE_SIZE = 416           # training resolution
LR = 0.004                 # base LR (per default 16-batch)

# OPTIMIZE = True   -> produces a smaller, faster model (recommended for edge deployment)
#                      Falls back automatically if the optimization is unstable.
# OPTIMIZE = False  -> skip optimization, keep the standard model.
OPTIMIZE = True

# ============================================================
# AUGMENTATION — training-time only, no inference-speed impact.
#   Multi-scale + rotation is the standard protocol for rotated detection.
#   Worth ~2-3 mAP at zero runtime cost.
# ============================================================
AUG = {
    "flip_prob":         0.75,        # horizontal + vertical + diagonal flips
    "color_photometric": True,        # brightness/contrast/saturation/hue
    "random_resize":     (0.5, 2.0),  # multi-scale ratio range
    "random_rotate":     True,        # free-angle rotation
}

classes = list(project.classes.keys())
project_name = project.name.lower().replace(" ", "_")

print(f"Model size:  {MODEL_SIZE}")
print(f"Classes:     {classes}")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Image size:  {IMAGE_SIZE}")
print(f"Optimize:    {OPTIMIZE}")
print(f"Augmentation keys: {list(AUG.keys())}")

## 4. Prepare Dataset

Convert the downloaded annotations into the pixel-space 4-corner format the training pipeline expects.

In [ ]:
import cv2
import numpy as np

DATA_ROOT = dataset.location


def convert_split(split):
    """Parse a split's COCO annotations and write pixel-space 4-corner OBB labels.

    Roboflow exports oriented boxes as a COCO `segmentation` polygon. The polygon
    may have many points (from the annotation tool's drawing). We fit a tight
    rotated rectangle via `cv2.minAreaRect` to recover the 4 OBB corners.
    """
    ann_file = os.path.join(DATA_ROOT, split, "_annotations.coco.json")
    if not os.path.exists(ann_file):
        return 0, 0, []
    with open(ann_file) as f:
        coco = json.load(f)

    id_to_img = {img["id"]: img for img in coco["images"]}

    img_anns = {}
    ann_counts = {}
    for ann in coco["annotations"]:
        img_anns.setdefault(ann["image_id"], []).append(ann)
        ann_counts[ann["category_id"]] = ann_counts.get(ann["category_id"], 0) + 1

    # Filter categories down to ones that actually have annotations.
    # Roboflow's COCO export injects a phantom workspace-slug category at
    # id 0 (zero annotations) that we don't want trained as a real class —
    # this drops it without us having to hard-code its slug.
    id_to_cat = {
        c["id"]: c["name"]
        for c in coco["categories"]
        if c["name"] != "__background__" and ann_counts.get(c["id"], 0) > 0
    }

    out_dir = os.path.join(DATA_ROOT, split, "labelTxt")
    os.makedirs(out_dir, exist_ok=True)

    n_imgs = n_anns = 0
    for img_id, img_info in id_to_img.items():
        stem = os.path.splitext(img_info["file_name"])[0]
        out_path = os.path.join(out_dir, stem + ".txt")
        lines = []
        for ann in img_anns.get(img_id, []):
            name = id_to_cat.get(ann["category_id"])
            if name is None:
                continue
            seg = ann.get("segmentation", [])
            if not (seg and isinstance(seg, list) and seg and len(seg[0]) >= 6):
                continue
            pts = np.array(seg[0], dtype=np.float32).reshape(-1, 2)
            rect = cv2.minAreaRect(pts)
            box = cv2.boxPoints(rect).flatten().tolist()
            lines.append(" ".join(f"{v:.2f}" for v in box) + f" {name} 0")
            n_anns += 1
        with open(out_path, "w") as f:
            f.write("\n".join(lines))
        n_imgs += 1

    ordered_classes = [id_to_cat[cid] for cid in sorted(id_to_cat.keys())]
    return n_imgs, n_anns, ordered_classes


classes = None
for split in ("train", "valid", "test"):
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.isdir(split_path):
        continue
    n_imgs, n_anns, split_classes = convert_split(split)
    if split == "train":
        classes = split_classes
    print(f"  {split}: {n_imgs} images, {n_anns} annotations")

assert classes, "No classes found — check train annotations"
print(f"\nFinal classes ({len(classes)}): {classes}")

## 5. Build Config & Train

In [ ]:
# Architecture configurations per model size.
# Checkpoint URLs are the published mmrotate RTMDet-R DOTA-v1 weights — they work as
# a starting point for custom data; the head is rebuilt for your class count anyway.
ARCH_CFG = {
    "tiny": {
        "deepen_factor": 0.167,
        "widen_factor":  0.375,
        "exp_on_reg":    False,
        "checkpoint":    "https://download.openmmlab.com/mmrotate/v1.0/rotated_rtmdet/rotated_rtmdet_tiny-3x-dota/rotated_rtmdet_tiny-3x-dota-9d821076.pth",
    },
    "small": {
        "deepen_factor": 0.33,
        "widen_factor":  0.5,
        "exp_on_reg":    False,
        "checkpoint":    "https://download.openmmlab.com/mmrotate/v1.0/rotated_rtmdet/rotated_rtmdet_s-3x-dota/rotated_rtmdet_s-3x-dota-11f6ccf5.pth",
    },
    "medium": {
        "deepen_factor": 0.67,
        "widen_factor":  0.75,
        "exp_on_reg":    True,
        "checkpoint":    "https://download.openmmlab.com/mmrotate/v1.0/rotated_rtmdet/rotated_rtmdet_m-3x-dota/rotated_rtmdet_m-3x-dota-beeadda6.pth",
    },
    "large": {
        "deepen_factor": 1.0,
        "widen_factor":  1.0,
        "exp_on_reg":    True,
        "checkpoint":    "https://download.openmmlab.com/mmrotate/v1.0/rotated_rtmdet/rotated_rtmdet_l-3x-dota/rotated_rtmdet_l-3x-dota-23992372.pth",
    },
}
assert MODEL_SIZE in ARCH_CFG, f"MODEL_SIZE must be one of {list(ARCH_CFG.keys())}"


def build_detector_config(cfg_arch, classes, data_root, epochs, batch_size, image_size, lr, aug):
    """Generate training config for custom DOTA-style data."""
    num_classes = len(classes)
    class_tuple = "(" + ", ".join(f'"{c}"' for c in classes) + (",)" if num_classes == 1 else ")")
    palette = "[" + ", ".join("(220, 20, 60)" for _ in classes) + "]"
    eta_min = lr * 0.05

    config = f"""default_scope = 'mmrotate'
custom_imports = dict(imports=['mmrotate.datasets', 'mmrotate.models'], allow_failed_imports=False)

default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=50),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=10, save_best='dota/mAP', rule='greater', max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='mmdet.DetVisualizationHook'),
)

custom_hooks = [
    dict(type='mmdet.EMAHook', ema_type='mmdet.ExpMomentumEMA', momentum=0.0002, update_buffers=True, priority=49),
]

env_cfg = dict(
    cudnn_benchmark=False,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)

vis_backends = [dict(type='LocalVisBackend')]
visualizer = dict(type='mmdet.DetLocalVisualizer', vis_backends=vis_backends, name='visualizer')
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=True)
log_level = 'INFO'
load_from = '{cfg_arch['checkpoint']}'
resume = False

angle_version = 'le90'

# --- Model ---
model = dict(
    type='mmdet.RTMDet',
    data_preprocessor=dict(
        type='mmdet.DetDataPreprocessor',
        mean=[103.53, 116.28, 123.675],
        std=[57.375, 57.12, 58.395],
        bgr_to_rgb=False,
        boxtype2tensor=False,
        batch_augments=None,
    ),
    backbone=dict(
        type='mmdet.CSPNeXt',
        arch='P5',
        expand_ratio=0.5,
        deepen_factor={cfg_arch['deepen_factor']},
        widen_factor={cfg_arch['widen_factor']},
        channel_attention=True,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU', inplace=True),
    ),
    neck=dict(
        type='mmdet.CSPNeXtPAFPN',
        in_channels=[int(256 * {cfg_arch['widen_factor']}), int(512 * {cfg_arch['widen_factor']}), int(1024 * {cfg_arch['widen_factor']})],
        out_channels=int(256 * {cfg_arch['widen_factor']}),
        num_csp_blocks=max(1, round(3 * {cfg_arch['deepen_factor']})),
        expand_ratio=0.5,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU', inplace=True),
    ),
    bbox_head=dict(
        type='RotatedRTMDetSepBNHead',
        num_classes={num_classes},
        in_channels=int(256 * {cfg_arch['widen_factor']}),
        stacked_convs=2,
        feat_channels=int(256 * {cfg_arch['widen_factor']}),
        angle_version=angle_version,
        anchor_generator=dict(type='mmdet.MlvlPointGenerator', offset=0, strides=[8, 16, 32]),
        bbox_coder=dict(type='DistanceAnglePointCoder', angle_version=angle_version),
        loss_cls=dict(type='mmdet.QualityFocalLoss', use_sigmoid=True, beta=2.0, loss_weight=1.0),
        loss_bbox=dict(type='RotatedIoULoss', mode='linear', loss_weight=2.0),
        with_objectness=False,
        exp_on_reg={cfg_arch['exp_on_reg']},
        share_conv=True,
        pred_kernel_size=1,
        use_hbbox_loss=False,
        scale_angle=False,
        loss_angle=None,
        norm_cfg=dict(type='SyncBN'),
        act_cfg=dict(type='SiLU', inplace=True),
    ),
    train_cfg=dict(
        assigner=dict(type='mmdet.DynamicSoftLabelAssigner', topk=13, iou_calculator=dict(type='RBboxOverlaps2D')),
        allowed_border=-1,
        pos_weight=-1,
        debug=False,
    ),
    test_cfg=dict(
        nms_pre=2000,
        min_bbox_size=0,
        score_thr=0.05,
        nms=dict(type='nms_rotated', iou_threshold=0.1),
        max_per_img=2000,
    ),
)

# --- Dataset ---
dataset_type = 'DOTADataset'
data_root = '{data_root}/'
CLASSES = {class_tuple}
metainfo = dict(classes=CLASSES, palette={palette})

backend_args = None

train_pipeline = [
    dict(type='mmdet.LoadImageFromFile', backend_args=backend_args),
    dict(type='mmdet.LoadAnnotations', with_bbox=True, box_type='qbox'),
    dict(type='ConvertBoxType', box_type_mapping=dict(gt_bboxes='rbox')),
    dict(type='mmdet.RandomResize', scale=({image_size}, {image_size}), ratio_range={aug['random_resize']}, resize_type='mmdet.Resize', keep_ratio=True),
    dict(type='mmdet.RandomCrop', crop_size=({image_size}, {image_size})),
    dict(type='mmdet.RandomFlip', prob={aug['flip_prob']}, direction=['horizontal', 'vertical', 'diagonal']),
    dict(type='RandomRotate', prob=0.5, angle_range=180, rect_obj_labels=None, rotate_type='mmrotate.Rotate') if {str(aug['random_rotate'])} else dict(type='mmdet.RandomFlip', prob=0.0),
    dict(type='mmdet.PhotoMetricDistortion') if {str(aug['color_photometric'])} else dict(type='mmdet.RandomFlip', prob=0.0),
    dict(type='mmdet.Pad', size=({image_size}, {image_size}), pad_val=dict(img=(128, 128, 128))),
    dict(type='mmdet.PackDetInputs'),
]

test_pipeline = [
    dict(type='mmdet.LoadImageFromFile', backend_args=backend_args),
    dict(type='mmdet.Resize', scale=({image_size}, {image_size}), keep_ratio=True),
    dict(type='mmdet.Pad', size=({image_size}, {image_size}), pad_val=dict(img=(128, 128, 128))),
    dict(type='mmdet.LoadAnnotations', with_bbox=True, box_type='qbox'),
    dict(type='ConvertBoxType', box_type_mapping=dict(gt_bboxes='rbox')),
    dict(type='mmdet.PackDetInputs', meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

train_dataloader = dict(
    batch_size={batch_size},
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    batch_sampler=None,
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        ann_file='train/labelTxt/',
        img_suffix='jpg',
        data_prefix=dict(img_path='train/'),
        filter_cfg=dict(filter_empty_gt=True),
        pipeline=train_pipeline,
        metainfo=metainfo,
    ),
)

val_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        data_root=data_root,
        ann_file='valid/labelTxt/',
        img_suffix='jpg',
        data_prefix=dict(img_path='valid/'),
        test_mode=True,
        pipeline=test_pipeline,
        metainfo=metainfo,
    ),
)

test_dataloader = val_dataloader
val_evaluator = dict(type='DOTAMetric', metric='mAP')
test_evaluator = val_evaluator

train_cfg = dict(type='EpochBasedTrainLoop', max_epochs={epochs}, val_interval=10)
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr={lr}, weight_decay=0.05),
    paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True),
)

param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False, begin=0, end=1000),
    dict(type='CosineAnnealingLR', eta_min={eta_min}, begin={epochs // 2}, end={epochs}, T_max={epochs - epochs // 2}, by_epoch=True, convert_to_iter_based=True),
]

auto_scale_lr = dict(base_batch_size=128)
"""
    return config


# ---------- build + train ----------
from mmengine.config import Config
from mmengine.runner import Runner
import mmrotate  # registers DOTADataset, RotatedRTMDetSepBNHead, etc.

os.makedirs("/content/configs", exist_ok=True)
config_path = f"/content/configs/oriented_{MODEL_SIZE}_{project_name}.py"

config_text = build_detector_config(
    ARCH_CFG[MODEL_SIZE], classes, DATA_ROOT,
    EPOCHS, BATCH_SIZE, IMAGE_SIZE, LR, AUG,
)
with open(config_path, "w") as f:
    f.write(config_text)
print(f"Config: {config_path}")

work_dir = f"/content/work_dirs/oriented_{MODEL_SIZE}_{project_name}"
cfg = Config.fromfile(config_path)
cfg.work_dir = work_dir
runner = Runner.from_cfg(cfg)
runner.train()
print(f"\nTraining complete. Work dir: {work_dir}")

## 6. Export & Optimize Model

In [ ]:
import cv2
import openvino as ov
import nncf
from mmdet.apis import init_detector

# Patch torch.onnx.export for compatibility in torch 2.3
import torch.onnx
if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        kwargs.pop("dynamo", None)
        return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True


# ---------- pick best checkpoint ----------
ckpts = sorted(glob.glob(os.path.join(work_dir, "best_*.pth")))
if not ckpts:
    ckpts = sorted(glob.glob(os.path.join(work_dir, "epoch_*.pth")))
assert ckpts, "No checkpoint found!"
checkpoint = ckpts[-1]
print(f"Checkpoint: {checkpoint}")

# Use EMA weights for export (better eval accuracy)
raw_ckpt = torch.load(checkpoint, map_location="cpu")
if "ema_state_dict" in raw_ckpt:
    raw_ckpt["state_dict"] = raw_ckpt["ema_state_dict"]
    print("Using averaged weights")
fixed_ckpt = os.path.join(work_dir, "best_for_export.pth")
torch.save(raw_ckpt, fixed_ckpt)

# ---------- Load model ----------
detector = init_detector(config_path, fixed_ckpt, device="cpu")
detector.eval()


# ---------- Export wrapper: unified rotated-box output tensor ----------
# Output shape: (batch, total_anchors, 5 + 1 + num_classes)
#               [x_center, y_center, width, height, angle_rad, obj_score, cls_1, ...]
#
# Rotated RTMDet head outputs 3 tensor lists:
#   cls_scores : (B, num_classes, H, W)
#   bbox_preds : (B, 4, H, W)   # l, t, r, b distances already multiplied by stride
#   angle_preds: (B, 1, H, W)   # angle prediction (radians, le90)
#
# Decode (matches DistanceAnglePointCoder):
#   w, h      = l + r, t + b
#   offset    = rotate([(r - l)/2, (b - t)/2], angle)
#   cx, cy    = anchor_point + offset

PRIOR_OFFSET = 0.0   # MUST match MlvlPointGenerator's `offset` in the training config


class DetectorExportWrapper(torch.nn.Module):
    def __init__(self, detector, num_classes, strides=(8, 16, 32), prior_offset=PRIOR_OFFSET):
        super().__init__()
        self.detector = detector
        self.num_classes = num_classes
        self.strides = strides
        self.prior_offset = prior_offset

    def forward(self, x):
        feats = self.detector.extract_feat(x)
        cls_scores, bbox_preds, angle_preds = self.detector.bbox_head(feats)
        outputs = []
        for cls_s, bbox_p, ang_p, stride in zip(cls_scores, bbox_preds, angle_preds, self.strides):
            B, _, H, W = cls_s.shape
            device = cls_s.device
            yv, xv = torch.meshgrid(
                torch.arange(H, device=device, dtype=torch.float32),
                torch.arange(W, device=device, dtype=torch.float32),
                indexing='ij',
            )
            grid = torch.stack((xv, yv), dim=-1)
            grid = (grid + self.prior_offset) * stride
            grid = grid.view(1, H * W, 2)
            bbox_p = bbox_p.permute(0, 2, 3, 1).reshape(B, H * W, 4)
            ang_p  = ang_p.permute(0, 2, 3, 1).reshape(B, H * W, 1)
            l, t, r, b = bbox_p[..., 0:1], bbox_p[..., 1:2], bbox_p[..., 2:3], bbox_p[..., 3:4]
            w = l + r
            h = t + b
            ox_local = (r - l) * 0.5
            oy_local = (b - t) * 0.5
            cos_a = torch.cos(ang_p)
            sin_a = torch.sin(ang_p)
            ox = cos_a * ox_local - sin_a * oy_local
            oy = sin_a * ox_local + cos_a * oy_local
            cx = grid[..., 0:1] + ox
            cy = grid[..., 1:2] + oy
            boxes = torch.cat([cx, cy, w, h, ang_p], dim=-1)
            cls_probs = cls_s.permute(0, 2, 3, 1).reshape(B, H * W, self.num_classes).sigmoid()
            obj = cls_probs.max(dim=-1, keepdim=True).values
            outputs.append(torch.cat([boxes, obj, cls_probs], dim=-1))
        return torch.cat(outputs, dim=1)


wrapped = DetectorExportWrapper(detector, num_classes=len(classes))
wrapped.eval()

os.makedirs("/content/export", exist_ok=True)
onnx_path = "/content/export/model.onnx"
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    wrapped, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB)")

# ---------- Standard-precision export ----------
core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "/content/export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin) / 1024 / 1024:.1f} MB)")

# ---------- Optimized model (if requested) ----------
MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32)
STD  = np.array([57.375, 57.12, 58.395], dtype=np.float32)

def preprocess_for_calibration(img_path, size):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(h * r), int(w * r)
    img = cv2.resize(img, (nw, nh))
    padded = np.full((size, size, 3), 128, dtype=np.uint8)
    padded[:nh, :nw] = img
    tensor = ((padded.astype(np.float32) - MEAN) / STD).transpose(2, 0, 1)[None]
    return tensor


opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    calib_imgs = sorted(glob.glob(f"{DATA_ROOT}/train/*.jpg"))[:200]
    calib_tensors = [t for t in (preprocess_for_calibration(p, IMAGE_SIZE) for p in calib_imgs) if t is not None]
    print(f"  Calibration samples: {len(calib_tensors)}")

    ov_model_for_opt = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_for_opt,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "/content/export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate) / 1024 / 1024:.1f} MB)")
    print(f"  Size reduction: {100 * (1 - os.path.getsize(opt_bin_candidate) / os.path.getsize(std_bin)):.0f}%")

    # ---- Detection-specific stability check ----
    # Check classification confidence, which SHOULD differ between zero input
    # (no objects) and a real image (objects → high conf).
    compiled_opt = core.compile_model(optimized, "CPU")
    compiled_std = core.compile_model(ov_model, "CPU")
    probe = calib_tensors[0]
    zeros = np.zeros_like(probe)

    def max_detection_score(out):
        # out shape: (1, N, 5+1+C) — peak score = obj * max_cls_prob
        arr = np.asarray(out)
        return float((arr[..., 5] * arr[..., 6:].max(axis=-1)).max())

    o_real_opt = list(compiled_opt([probe]).values())[0]
    o_zero_opt = list(compiled_opt([zeros]).values())[0]
    o_real_std = list(compiled_std([probe]).values())[0]

    real_opt = max_detection_score(o_real_opt)
    zero_opt = max_detection_score(o_zero_opt)
    real_std = max_detection_score(o_real_std)

    drift = abs(real_opt - real_std) / max(real_std, 1e-8)
    print(f"  Stability check: real_conf={real_opt:.3f}, zero_conf={zero_opt:.3f}, drift={drift*100:.1f}%")

    if real_opt > 0.1 and drift < 0.50:
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK — packaging optimized version")
    else:
        print("  Optimization unstable — packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} — packaging standard version only.")

## 7. Save & Download Pickle

In [ ]:
# Pickle format mirrors the OD pickle:
#   { bin, xml, cls, colors, meta }
# meta.type is 'rod' (oriented bounding box) so the inference side can dispatch.

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": classes,
    "colors": project.colors,
    "meta": {
        "model_type": MODEL_SIZE,
        "type": "rod",
        "image_size": IMAGE_SIZE,
        "precision": precision_tag,
    },
}

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Pickle contains {variant} model ({len(bin_data) / 1024 / 1024:.1f} MB)")
print(f"\nSaved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Model size: {MODEL_SIZE}")
print(f"Classes: {classes}")
print(f"Image size: {IMAGE_SIZE}")

files.download(pickle_path)